# CAD Aero Table Generation & Surrogate Training

This notebook generates DOLFINx-based aerodynamic coefficient tables from normalized CAD STL meshes and trains GPR surrogates for use in Project Icarus.

**Features:**
- Select vehicles from the CAD manifest (`kh101`, `ss18_satan`, `topol_m`, `df41`)
- Quality presets: `draft`, `standard`, `high`, `production`
- Custom sweep grid control (Mach, alpha, beta, altitude, control deflection)
- Real-time progress tracking and timing
- Automatic surrogate training, validation, and persistence
- Outputs saved to `reference/vehicles/<key>/`

---

## 1. Setup & Imports

In [1]:
import sys
from pathlib import Path

# In VS Code / Jupyter, CWD may differ from the notebook location.
# Search upward from CWD for the project root (directory that contains
# the 'project_icarus' package) and add it to sys.path if missing.
_cwd = Path.cwd()
_project_root = None
for _candidate in [_cwd, *_cwd.parents]:
    if (_candidate / "project_icarus").is_dir():
        _project_root = _candidate
        break

if _project_root is None:
    _project_root = _cwd

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

print(f"Project root: {_project_root}")
print(f"sys.path[0]: {sys.path[0]}")


Project root: /mnt/user/public/project-icarus
sys.path[0]: /mnt/user/public/project-icarus


In [2]:
import os
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import h5py
import joblib
from tqdm.notebook import tqdm

# Project Icarus imports
from project_icarus.aero.cfd_generators import (
    SweepSpec,
    run_sweep,
    save_sweep_hdf5,
)
from project_icarus.aero.geometry import get_vehicle
from project_icarus.aero.stl_loader import CAD_MANIFEST, load_cad_vehicle, surface_area
from project_icarus.surrogates.train_gpr import (
    MultiOutputGPR,
    default_model_path,
    default_h5_path,
    train_vehicle_gpr,
    load_vehicle_gpr,
)

# Output directories
H5_DIR = Path("reference/vehicles")
MODEL_DIR = Path("reference/vehicles")
H5_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK")
print(f"CAD manifest vehicles: {sorted(CAD_MANIFEST.keys())}")

Imports OK
CAD manifest vehicles: ['df41', 'kh101', 'ss18_satan', 'topol_m']


## 2. Configuration

Choose a quality preset or define a custom sweep grid.

| Preset | Mach | α | β | Altitude | Δ | Description |
|---|---|---|---|---|---|---|
| `draft` | 5 pts | 3 pts | 1 pt | 2 pts | 1 pt | Quick validation (~4–8 min/vehicle) |
| `standard` | 6 pts | 5 pts | 3 pts | 4 pts | 1 pt | Balanced fidelity (~15–30 min/vehicle) |
| `high` | 10 pts | 9 pts | 5 pts | 6 pts | 3 pts | Detailed table (~45–90 min/vehicle) |
| `production` | 12 pts | 11 pts | 7 pts | 8 pts | 5 pt | Full envelope (>2–4 h/vehicle) |

In [ ]:
# Quality presets
QUALITY_PRESETS = {
    "draft": {
        "mach_range": (0.5, 5.0, 5),
        "alpha_range": (-5.0, 5.0, 3),
        "beta_range": (0.0, 0.0, 1),
        "altitude_range": (0.0, 50e3, 2),
        "delta_range": (0.0, 0.0, 1),
    },
    "standard": {
        "mach_range": (0.5, 5.0, 6),
        "alpha_range": (-10.0, 10.0, 5),
        "beta_range": (-5.0, 5.0, 3),
        "altitude_range": (0.0, 100e3, 4),
        "delta_range": (0.0, 0.0, 1),
    },
    "high": {
        "mach_range": (0.3, 6.0, 10),
        "alpha_range": (-15.0, 15.0, 9),
        "beta_range": (-10.0, 10.0, 5),
        "altitude_range": (0.0, 150e3, 6),
        "delta_range": (0.0, 15.0, 3),
    },
    "production": {
        "mach_range": (0.3, 6.0, 12),
        "alpha_range": (-10.0, 20.0, 11),
        "beta_range": (-5.0, 5.0, 7),
        "altitude_range": (0.0, 150e3, 8),
        "delta_range": (0.0, 15.0, 5),
    },
}

# --- USER SETTINGS -------------------------------------------------
QUALITY = "standard"          # draft | standard | high | production
VEHICLE_KEYS = ["kh101", "ss18_satan", "topol_m", "df41"]
MAX_PANELS = 500_000            # panel cap for mesh-backed backends
TRAIN_SURROGATE = True          # train GPR after sweep?
# -------------------------------------------------------------------

grid = QUALITY_PRESETS[QUALITY]
print(f"Quality preset: {QUALITY}")
print(f"Vehicles: {VEHICLE_KEYS}")
print(f"Grid points per sweep: {np.prod([len(np.linspace(*v)) for v in grid.values()]):.0f}")

## 3. Vehicle & Mesh Inspection

Quick sanity checks on the STL-loaded meshes before launching long sweeps.

In [ ]:
def inspect_vehicle(key: str) -> None:
    info = CAD_MANIFEST[key]
    g = get_vehicle(info.proxy_for)
    print(f"\n=== {key} -> {g.name} ===")
    print(f"  Preset length: {g.body_length:.2f} m, diameter: {g.body_diameter:.2f} m")
    print(f"  STL file: {info.filename} (scale={info.scale})")
    
    v, f = load_cad_vehicle(key)
    z_ext = float(v[:, 2].max() - v[:, 2].min())
    xy_ext = float(np.linalg.norm(v[:, :2].max(axis=0) - v[:, :2].min(axis=0)))
    area = surface_area(v, f)
    centroid = v.mean(axis=0)
    
    print(f"  STL vertices: {v.shape[0]}, faces: {f.shape[0]}")
    print(f"  Z extent: {z_ext:.3f} m (preset: {g.body_length:.2f} m, ratio: {z_ext/g.body_length:.2f})")
    print(f"  XY radius: {xy_ext/2:.3f} m (preset: {g.body_diameter/2:.3f} m, ratio: {(xy_ext/2)/(g.body_diameter/2):.2f})")
    print(f"  Surface area: {area:.2f} m²")
    print(f"  Centroid: [{centroid[0]:.4f}, {centroid[1]:.4f}, {centroid[2]:.4f}] m")

for key in tqdm(VEHICLE_KEYS, desc="Inspecting meshes"):
    inspect_vehicle(key)

## 4. Sweep Execution

Run the CFD sweep for each selected vehicle using the DOLFINx backend.

> **Note:** First-time runs are slower due to DOLFINx compilation. Subsequent runs benefit from caching.

In [ ]:
def run_sweep_for_vehicle(key: str, grid: dict) -> dict:
    info = CAD_MANIFEST[key]
    g = get_vehicle(info.proxy_for)
    
    print(f"\n[{datetime.now():%H:%M:%S}] Sweeping {key} ({g.name})")
    
    t0 = time.time()
    spec = SweepSpec(
        vehicle=key,
        backend="dolfinx",
        **grid,
    )
    n_points = int(np.prod([len(np.linspace(*v)) for v in grid.values()]))
    pbar = tqdm(total=n_points, desc=f"Sweeping {key}", leave=False)
    def _prog(done, total):
        pbar.n = done
        pbar.refresh()
    result = run_sweep(spec, progress=_prog)
    pbar.close()
    elapsed = time.time() - t0
    
    n = result["coeffs"].shape[0]
    cd_min = float(result["coeffs"][:, 0].min())
    cd_max = float(result["coeffs"][:, 0].max())
    print(f"  {n} points in {elapsed:.1f}s ({elapsed/n:.2f}s/point)")
    print(f"  Cd range: [{cd_min:.4f}, {cd_max:.4f}]")
    print(f"  All finite: {np.all(np.isfinite(result['coeffs']))}")
    
    return {"key": key, "result": result, "elapsed": elapsed}


sweep_results = []
for key in tqdm(VEHICLE_KEYS, desc="Sweeping vehicles"):
    try:
        sweep_results.append(run_sweep_for_vehicle(key, grid))
    except Exception as exc:
        print(f"  FAILED {key}: {exc}")
        sweep_results.append({"key": key, "error": str(exc)})

## 5. Save HDF5 Tables

In [ ]:
def save_result(row: dict) -> str | None:
    if "error" in row:
        return None
    key = row["key"]
    info = CAD_MANIFEST[key]
    vehicle = info.proxy_for
    h5_path = H5_DIR / f"aero.h5"
    save_sweep_hdf5(row["result"], str(h5_path))
    print(f"  Saved: {h5_path}")
    return str(h5_path)

h5_paths = {}
for row in tqdm(sweep_results, desc="Saving HDF5 tables"):
    path = save_result(row)
    if path:
        h5_paths[row["key"]] = path

print(f"\nHDF5 tables saved: {len(h5_paths)}/{len(sweep_results)}")

## 6. Train GPR Surrogates

Train one multi-output GPR per vehicle. The model predicts `[Cd, Cy, Cm, Cn, Cl_roll]` given `[Mach, alpha, beta, altitude, delta]`.

In [ ]:
def train_and_validate(key: str, h5_path: str) -> dict:
    info = CAD_MANIFEST[key]
    vehicle = info.proxy_for
    
    print(f"\nTraining surrogate for '{vehicle}' ({key})...")
    t0 = time.time()
    
    model = train_vehicle_gpr(vehicle)
    model_path = default_model_path(vehicle, base_dir=str(MODEL_DIR))
    
    elapsed = time.time() - t0
    X_test = np.array([
        [1.0, 0.0, 0.0, 10e3, 0.0],
        [2.0, 3.0, 0.0, 30e3, 0.0],
        [3.0, -2.0, 1.0, 50e3, 0.0],
    ])
    pred = model.predict(X_test)
    
    print(f"  Model saved: {model_path}")
    print(f"  Training time: {elapsed:.1f}s")
    print(f"  Validation predictions (first 3 test points):")
    coeff_names = ["Cd", "Cy", "Cm", "Cn", "Cl_roll"]
    for i, row in enumerate(pred):
        print(f"    Point {i+1}: " + ", ".join(f"{c}={v:.4f}" for c, v in zip(coeff_names, row)))
    
    return {"key": key, "vehicle": vehicle, "model_path": model_path, "time": elapsed}


if TRAIN_SURROGATE:
    surrogate_results = []
    for key, h5_path in tqdm(h5_paths.items(), desc="Training surrogates"):
        try:
            surrogate_results.append(train_and_validate(key, h5_path))
        except Exception as exc:
            print(f"  FAILED {key}: {exc}")
            surrogate_results.append({"key": key, "error": str(exc)})
else:
    print("Surrogate training skipped (TRAIN_SURROGATE=False)")

## 7. Summary & Verification

In [ ]:
def verify_surrogate(key: str) -> None:
    info = CAD_MANIFEST[key]
    vehicle = info.proxy_for
    model_path = MODEL_DIR / f"surrogate.pkl"
    h5_path = H5_DIR / f"aero.h5"
    
    print(f"\n=== {key} -> {vehicle} ===")
    
    if not model_path.exists():
        print(f"  Model missing: {model_path}")
        return
    
    model = joblib.load(str(model_path))
    print(f"  Model loaded: {type(model).__name__}")
    
    with h5py.File(str(h5_path), "r") as hf:
        print(f"  HDF5 datasets: {list(hf.keys())}")
        print(f"  Coeffs shape: {hf['coeffs'].shape}")
        
    X = np.array([[1.5, 2.5, 0.0, 10e3, 0.0]])
    pred = model.predict(X)
    coeff_names = ["Cd", "Cy", "Cm", "Cn", "Cl_roll"]
    print(f"  Sanity prediction: " + ", ".join(f"{c}={v:.4f}" for c, v in zip(coeff_names, pred[0])))


print("\n" + "="*60)
print("VERIFICATION SUMMARY")
print("="*60)

all_keys = [r["key"] for r in sweep_results if "error" not in r]
for key in tqdm(VEHICLE_KEYS, desc="Verifying surrogates"):
    if key in all_keys:
        verify_surrogate(key)
    else:
        print(f"\n=== {key} ===")
        print(f"  SKIPPED / FAILED")

---

## Notes

- **Output convention:** HDF5 tables → `reference/vehicles/<key>/aero.h5`
- **Output convention:** GPR pickles → `reference/vehicles/<key>/surrogate.pkl`
- **CAD normalization:** Meshes are re-centered and aligned to +Z automatically via `stl_loader.py`.
- **Mesh backend:** `build_surface_mesh(backend="stl")` is used automatically when a preset has `stl_alias` set.
- **Validation:** All generated tables and surrogates are verified for finite coefficients and valid shapes.

For custom vehicles, add entries to `project_icarus/aero/stl_loader.py` → `CAD_MANIFEST` and set `stl_alias` in `project_icarus/aero/geometry.py` → `VEHICLE_PRESETS`.